# LELA60331 Week 3 Seminar: Vector-based models of word meaning

This week we are going to look at vector-based models of word meaning. I am first of all going to have to introduce a Python library called Numpy (https://numpy.org/devdocs/user/absolute_beginners.html).

### Numpy

Numpy is widely used for representing and processing arrays, including multidimensional arrays (known to us as Vectors/Matrices/Tensors). It is fast, intuitive and has lots of helpful built-in functions (we will make use of some of these later in the semester).

To use numpy we need to import it as follows. The naming of numpy as np is a widely-used convention.

This sheet also uses some Python from the **Week 3 Python workbook (LELA60341)**: writing functions with `def`, `enumerate`, list comprehensions and dictionary comprehensions.

In [ ]:
import numpy as np

We can create empty numpy arrays as follows:

In [ ]:
# For a 1-dimensional array
np.zeros(4)

In [ ]:
# For a 2 dimensional array
np.zeros((4, 5))

We can also create them from Python lists as follows:

In [ ]:
# Example vector
np.array([9,2,3,5])

In [ ]:
# Example rank 2 tensor (specifically a 2x4 matrix)
np.array([[9, 2, 3, 5], [4, 6, 7, 3]])

In [ ]:
# Example rank 3 tensor 3x2x4
np.array([[[0, 1, 2, 3],[4, 5, 6, 7]],[[0, 1, 2, 3],[4, 5, 6, 7]],[[0 ,1 ,2, 3],[4, 5, 6, 7]]])

The arrays must be rectangular, not ragged, or you will see an error (a `ValueError`):

In [ ]:
# A ragged array: the last row is longer than the others, so this gives an error
np.array([[9, 2, 3, 5], [4, 6, 7, 3], [5, 7, 1, 2, 7]])

Just as with Python lists we can use indices to find individual values:

In [ ]:
a=np.array([9,2,3,5])
a[1]

And ranges:

In [ ]:
a[1:3]

We can do the same for multidimensional arrays. Indexes should be in the order of nesting. So for a rank 2 tensor the row index comes first and the column second:

In [ ]:
a = np.array([[9, 2, 3, 5],
              [4, 6, 7, 3],
              [5, 7, 1, 2]])
a[1, 0]

We can assign values to particular positions in our tensor using indices:

In [ ]:
a[0,0] = 1000
a[2,1] = 2000
print(a)

For vectors we can perform the operations that we learned about in our lecture as follows:

In [ ]:
# Vector addition
a = np.array([9, 2, 3, 5])
b = np.array([1, 2, 3, 4])
c = a + b
print(a)
print(b)
print(c)

In [ ]:
# Vector subtraction
a = np.array([9, 2, 3, 5])
b = np.array([1, 2, 3, 4])
c = a - b
print(a)
print(b)
print(c)

In [ ]:
# Dot product: multiply the vectors element by element, then add up the results
a = np.array([9, 2, 3, 5])
b = np.array([1, 2, 3, 4])
c = a * b
dp = sum(c)
print(a)
print(b)
print(c)
print(dp)

**Activity 1 (cosine as a function)**

The cosine of the angle between two vectors is their dot product divided by the product of their lengths:

$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|} \qquad \text{where } \|a\| = \sqrt{a \cdot a}$$

We will need the cosine many times in this sheet, so we write it as a **function** (Week 3 Python workbook, Part 1). Fill in the blanks. `np.sqrt()` gives a square root.

In [ ]:
def cosine(a, b):
    dot = ???                   # the dot product of a and b (see the cell above)
    length_a = ???              # the square root of the dot product of a with itself
    length_b = ???
    return ???

a = np.array([9, 2, 3, 5])
b = np.array([1, 2, 3, 4])
print(cosine(a, b))
print(cosine(a, a))             # a vector always has cosine 1 with itself

### Building Word Vectors

In [ ]:
# download the book from Project Gutenberg (the ! runs a command outside Python)
!wget https://www.gutenberg.org/files/2554/2554-0.txt
# (In Jupyter rather than Colab, download it with:
#  from urllib.request import urlretrieve
#  urlretrieve("https://www.gutenberg.org/files/2554/2554-0.txt", "2554-0.txt"))

with open('2554-0.txt') as fp:
    c_and_p = fp.read()
# keep the novel itself: from the first chapter heading up to Project Gutenberg's end-of-book notice
start = c_and_p.find('CHAPTER I\n')
end = c_and_p.find('*** END OF THE PROJECT GUTENBERG')
text = c_and_p[start:end].lower().replace('\n', ' ')
# keep only the letters a-z and spaces (as in Week 2), then split the text into a list of words
allowed = 'abcdefghijklmnopqrstuvwxyz '
text = ''.join([ch for ch in text if ch in allowed])
words = text.split()

In [ ]:
words[:10]

**Activity 2 (to be tackled together): a word-by-context matrix**

Each **row** of `M` stands for one word type, and each **column** for a *context* word. `M[i, j]` should count how often context word `j` occurs within `window_size` words of word `i`.

* Every word type gets a row, but only the 1000 most frequent words are used as contexts (columns). Rare context words add little information, and a full 10,000 x 10,000 matrix would take up about 800 MB of memory. With 1000 columns it needs about 80 MB.
* `word_to_index` and `context_to_index` map each word to its row and to its column. They are built with dictionary comprehensions and `enumerate` (Week 3 Python workbook). Looking words up in a dictionary is far faster than `type_list.index(word)`, which searches the whole list each time.
* For each word in the text, the inner loop visits the positions `j` around it. The first two conditions check that `j` is a real position and not the word itself. Fill in the third (the word at position `j` must be one of our contexts) and the two indices.

In [ ]:
from collections import Counter

token_count = len(words)
# The vocabulary: each word type once. Sorting makes the order the same every time.
type_list = sorted(set(words))
# The type count is the number of unique words. The token count is the total number of words including repetitions.
type_count = len(type_list)
# The contexts: the 1000 most frequent words
context_list = [word for word, count in Counter(words).most_common(1000)]
# Which row belongs to each word, and which column belongs to each context word
word_to_index = {word: i for i, word in enumerate(type_list)}
context_to_index = {word: j for j, word in enumerate(context_list)}
# We create a matrix in which to store the co-occurrence counts
M = np.zeros((type_count, len(context_list)))
window_size = 2

for i, word in enumerate(words):
    for j in range(i - window_size, i + window_size + 1):
        if 0 <= j < token_count and j != i and ???:      # ... and the word at position j is a context word
            M[???, ???] += 1                               # the row for word, the column for the word at position j

**Activity 3**

Use your `cosine` function to calculate the cosine between "walk" and "run", and between "walk" and "shine". What does the outcome tell us?

In [ ]:
walk_vec = M[word_to_index["walk"]]       # the row of M for "walk"
run_vec = ???
shine_vec = ???

print("walk ~ run:  ", ???)
print("walk ~ shine:", ???)

### Pretrained embeddings

Vectors are best when learned from very large text collections. However learning such vectors, particular using neural network methods rather than simple counting, is very computationally intensive. As a result most people make use of pretrained embeddings such as those found at

https://code.google.com/archive/p/word2vec/

or

https://nlp.stanford.edu/projects/glove/

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
#from urllib.request import urlretrieve
#url = "http://nlp.stanford.edu/data/glove.6B.zip"
#filename="glove.6B.zip"
#urlretrieve(url, filename)
!unzip -q glove.6B.zip

In [ ]:
embedding_file = 'glove.6B.100d.txt'
# The file lists 400,000 words, most frequent first. To save memory and time we only keep the first 100,000.
max_words = 100000
type_list = []
embeddings = []
with open(embedding_file) as fp:
    for line in fp:
        fields = line.split()                                        # the word, then 100 numbers
        type_list.append(fields[0])
        embeddings.append(np.array([float(x) for x in fields[1:]]))  # turn the numbers from strings into floats
        if len(type_list) == max_words:
            break
M = np.array(embeddings)
word_to_index = {word: i for i, word in enumerate(type_list)}

In [ ]:
w1 = "football"
w2 = "rugby"
w3 = "cricket"
w1_vec = M[word_to_index[w1]]
w2_vec = M[word_to_index[w2]]
w3_vec = M[word_to_index[w3]]

**Activity 4**

Use your `cosine` function to calculate the cosine between each pair of the words above. What do the cosine values tell us?

In [ ]:
print(w1, w2, ???)
print(w1, w3, ???)
print(w2, w3, ???)

### Finding the most similar words

One thing we often want to do is to find the most similar words to a given word/vector. An exhaustive N x N comparison is very time consuming, and so we can make use of an efficient "nearest neighbours" finding algorithm. We are just using this algorithm here so we won't go into it in any detail. We use the implementation in the Scikitlearn toolkit, which we will learn more about in the Python programming sessions in RM in CCL 2.

In [ ]:
from sklearn.neighbors import NearestNeighbors
nbrs = NearestNeighbors(n_neighbors=5, algorithm='ball_tree').fit(M)

In [ ]:
w = "football"
w_vec = M[word_to_index[w]]
distances, indices = nbrs.kneighbors([w_vec])   # the positions of the 5 nearest vectors
for i in indices[0]:
    print(type_list[i])

**Activity 5 (a `most_similar` function)**

We will look up neighbours many times, so let's wrap the cell above in a function (Week 3 Python workbook).

1. Complete the list comprehension so that the function returns the neighbouring **words** rather than their positions.
2. If the word isn't in the vocabulary, `word_to_index[word]` would raise a `KeyError`. Fill in the `if` condition so that the function returns an empty list instead.
3. Try it with a few words of your own, and with `n=10`.

In [ ]:
def most_similar(word, n=5):
    if ???:                                  # the word is not in our vocabulary
        return []
    vec = M[word_to_index[word]]
    distances, indices = nbrs.kneighbors([vec], n_neighbors=n)
    return [??? for i in indices[0]]

print(most_similar("football"))
print(most_similar("rugby", n=10))
print(most_similar("xyzzy"))

**Activity 6**

Use `most_similar` to find some examples where the system fails, and explain why you think it has done so.

### Analogical reasoning

Another semantic property of embeddings is their ability to capture relational meanings. In an important early vector space model of cognition, Rumelhart and Abrahamson (1973) proposed the parallelogram model for solving simple analogy problems of the form a is to b as a* is to what?. In such problems, a system is given a problem like apple:tree::grape:?, i.e., apple is to tree as grape is to ___, and must fill in the word vine.

In the parallelogram model, the vector from the word apple to the word tree (= tree − apple) is added to the vector for grape (grape); the nearest word to that point is returned.

**Activity 7 (analogies)**

Complete the function below so that it solves analogy problems of the form a : b :: c : ?.

1. `spatial_relationship` is the vector from `a` to `b` (that is, b − a).
2. `target_vec` is that relationship added to the vector for `c`.
3. The nearest neighbours of `target_vec` usually include the three input words themselves, so the last line uses a list comprehension with an `if` to leave them out.

Then come up with an analogy problem of your own and use the function to solve it.

In [ ]:
def analogy(a, b, c, n=5):
    a_vec = M[word_to_index[a]]
    b_vec = M[word_to_index[b]]
    c_vec = M[word_to_index[c]]
    spatial_relationship = ???
    target_vec = ???
    distances, indices = nbrs.kneighbors([target_vec], n_neighbors=n + 3)
    neighbours = [type_list[i] for i in indices[0]]
    return [w for w in neighbours if w not in (a, b, c)][:n]

print(analogy("apple", "tree", "grape"))